# Task 3 - Multi-Agent Financial Research System

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indeewara/CDAZZDEV-MLE-IndeewaraJayasuriya/blob/main/task3_agentic/task3_multi_agent_research.ipynb)

Agents that research a stock with five tools, decide each step from what they observe, hand structured data to each
other, remember context, and log every tool call.

| Part | Built with | Code |
|---|---|---|
| **3A** Tool-using research agent | LangGraph loop (agent ⇄ tools), `gpt-oss-120b` on Groq, one tool call per step | `tools.py`, `agent.py` |
| **3B** Two-agent pipeline | Data Analyst + Research Writer with enforced tool access, Pydantic hand-offs, critique loop | `multi_agent.py` |
| **3C** Memory and observability | LangGraph `MemorySaver` (short-term), JSON cache keyed by ticker+date (persistent), `logs/agent_trace.jsonl` | `memory.py`, `tracing.py` |

**Run:** locally the Groq key comes from `.env`; on Colab add it as the Colab Secret `GROQ_API_KEY`. No GPU needed.
A full run uses ~130k tokens of `gpt-oss-120b`'s 200k free daily quota; if the quota runs out mid-run the agents fall
back to `gpt-oss-20b` (logged), with lower report quality.

## 0. Setup

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/indeewara/CDAZZDEV-MLE-IndeewaraJayasuriya.git"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def sh(cmd):
    # run a command and SHOW its output if it fails - Colab otherwise hides subprocess output
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:])
        print(r.stderr[-3000:])
        raise RuntimeError("command failed: " + " ".join(cmd[:6]))
    return r

if IN_COLAB:
    if not os.path.exists("CDAZZDEV-MLE-IndeewaraJayasuriya"):
        sh(["git", "clone", "-q", REPO_URL])
    else:
        sh(["git", "-C", "CDAZZDEV-MLE-IndeewaraJayasuriya", "pull", "-q"])
    os.chdir("CDAZZDEV-MLE-IndeewaraJayasuriya/task3_agentic")
    sh([sys.executable, "-m", "pip", "install", "-q", "langgraph==1.2.14", "langchain-groq==1.1.3",
        "langchain-core==1.6.7", "ddgs==9.16.0", "yfinance", "feedparser", "groq", "python-dotenv", "pydantic"])
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

sys.path.insert(0, os.getcwd())
commit = subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print("working dir: .../" + os.path.basename(os.getcwd()), "| Colab:", IN_COLAB, "| code version:", commit)

In [ ]:
import json, logging
import pandas as pd
from IPython.display import Markdown, display

logging.disable(logging.WARNING)          # silence yfinance/library chatter; the agent trace is printed below
pd.set_option("display.max_colwidth", 90)

import tracing
from tools import TOOLS
from agent import AGENT_MODEL, MAX_TOOL_ROUNDS, run_research
from multi_agent import DATA_ANALYST_TOOLS, RESEARCH_WRITER_TOOLS, run_pipeline
from memory import ask_followup, cache_path, load_report, research_with_cache, save_report

tracing.VERBOSE = True
print("agent model:", AGENT_MODEL, "| fallback:", tracing.MODEL_FALLBACKS[AGENT_MODEL], "| tool budget:", MAX_TOOL_ROUNDS)
print("trace file:", tracing.TRACE_PATH.relative_to(tracing.ROOT))

def trace(session=None):
    rows = [json.loads(l) for l in tracing.TRACE_PATH.read_text().splitlines()]
    return [r for r in rows if session is None or r.get("session_id") == session]

---
# 3A - Tool-Using Research Agent

## The five tools
Each tool returns a compact JSON-serialisable `dict`; a failure returns `{"error": ...}` instead of raising, so the
agent can observe it and try something else. Price, indicator and news code is reused from Task 1 (already tested).

In [ ]:
for name, t in TOOLS.items():
    print(f"{name}({', '.join(t.args)})\n    {t.description.splitlines()[0]}")

Each tool called once directly, to show it is callable and what type it returns:

In [ ]:
tracing.new_session()
samples = {
    "get_price_data": {"ticker": "AAPL", "period": "1y"},
    "calculate_volatility": {"ticker": "AAPL", "window": 30},
    "get_news": {"ticker": "AAPL", "n": 5},
    "web_search": {"query": "Apple stock analyst price target"},
}
outputs = {name: TOOLS[name].invoke(args) for name, args in samples.items()}
outputs["llm_sentiment"] = TOOLS["llm_sentiment"].invoke({"headlines": [h["title"] for h in outputs["get_news"]["headlines"]]})
display(pd.DataFrame([{"tool": n, "returns": type(o).__name__, "keys": ", ".join(list(o)[:7]),
                       "error": o.get("error")} for n, o in outputs.items()]))

## Autonomous research run
The query from the brief. The agent sees the tools' descriptions and decides every step itself - there is no fixed
order in the code. Each step prints what the model was thinking (`thinks`), the tool it chose (`decides`) and the
result it observed; then the report writer turns the observations into the three required sections and checks every
number in the report against the tool results.

In [ ]:
report_a, run_a = run_research("AAPL")

In [ ]:
display(Markdown(report_a.to_markdown()))
print("numbers in the report not traceable to a tool result:", run_a["ungrounded"] or "none")

### Observe → decide, step by step
Each row is one decision: what the agent knew at that point, and the call it made next.

In [ ]:
from langchain_core.messages import AIMessage, ToolMessage

def decision_table(messages):
    rows, step = [], 0
    for m in messages:
        if isinstance(m, AIMessage) and m.tool_calls:
            step += 1
            c = m.tool_calls[0]
            rows.append({"step": step, "agent's reasoning": (m.additional_kwargs.get("reasoning_content") or "")[:140],
                         "decided to call": f"{c['name']}({json.dumps(c['args'])[:70]})"})
        elif isinstance(m, ToolMessage) and rows:
            rows[-1]["observed"] = ("ERROR: " if '"error"' in m.content[:12] else "") + m.content[:90]
    return pd.DataFrame(rows).set_index("step")

display(decision_table(run_a["state"]["messages"]))

## Error handling: a tool outage
`get_news` is forced to fail for this run (`tracing.FAILURE_INJECTION`). The agent is not told what to do about it;
it has to notice the error and find another way to get the information.

In [ ]:
tracing.FAILURE_INJECTION.add("get_news")
try:
    report_err, run_err = run_research("NVDA")
finally:
    tracing.FAILURE_INJECTION.clear()

In [ ]:
display(decision_table(run_err["state"]["messages"]))
print("report produced despite the outage:", report_err is not None,
      "| sources used:", report_err.data_sources if report_err else None)

---
# 3B - Multi-Agent Coordination

| Agent | Role | Tools (enforced) | Output |
|---|---|---|---|
| **A - Data Analyst** | quantitative analysis | get_price_data, calculate_volatility, llm_sentiment | `DataBrief` (Pydantic) |
| **B - Research Writer** | qualitative synthesis | web_search, get_news | `MultiAgentReport` (Pydantic) |

Neither agent can produce news sentiment alone: B has the headlines but no sentiment tool; A has the sentiment tool
but cannot fetch news. Tool access is enforced in the graph - a call to another agent's tool is refused - not just
stated in the prompt.

```
A: price + volatility --DataBrief--> B: news + web search --ClarificationRequest--> A: llm_sentiment
                                     B: final report <--ClarificationResponse-------/
```

**Enforcement check** (a scripted model tries to use the other agent's tool; no API call):

In [ ]:
from langchain_core.messages import HumanMessage
from agent import build_agent

class ScriptedLLM:
    def __init__(self, replies): self.replies = list(replies)
    def bind_tools(self, tools, **_): return self
    def invoke(self, messages): return self.replies.pop(0)

def attempt(agent_name, tool_names, forbidden):
    llm = ScriptedLLM([AIMessage(content="", tool_calls=[{"name": forbidden, "args": {"ticker": "AAPL"}, "id": "x", "type": "tool_call"}]),
                       AIMessage(content="done")])
    g = build_agent([TOOLS[n] for n in tool_names], "sys {max_rounds}", name=agent_name, llm=llm)
    out = g.invoke({"messages": [HumanMessage("go")], "tool_rounds": 0}, {"recursion_limit": 20})
    return next(m.content for m in out["messages"] if isinstance(m, ToolMessage))

tracing.VERBOSE = False
print("Research Writer tries get_price_data ->", attempt("research_writer", RESEARCH_WRITER_TOOLS, "get_price_data"))
print("Data Analyst tries get_news         ->", attempt("data_analyst", DATA_ANALYST_TOOLS, "get_news"))
tracing.VERBOSE = True

## Full pipeline run - the complete message trace
Everything below is printed live: each agent's reasoning, decisions and tool calls, and every hand-off as the
validated JSON object that was passed.

In [ ]:
report_b, run_b = run_pipeline("MSFT")

In [ ]:
display(Markdown(report_b.to_markdown()))
print("numbers not traceable to a tool result:", run_b["ungrounded"] or "none",
      "| Data Analyst's answer used in the analysis:", run_b["incorporated"])

Hand-offs for this run, read back from `agent_trace.jsonl`:

In [ ]:
display(pd.DataFrame([{"from": e["from"], "to": e["to"], "schema": e["schema"],
                       "payload (start)": json.dumps(e["payload"])[:110]}
                      for e in trace(run_b["session_id"]) if e["event"] == "handoff"]))
by_agent = {}
for e in trace(run_b["session_id"]):
    if e["event"] == "tool_call":
        by_agent.setdefault(e["agent"], []).append(e["tool"])
print("tools actually called per agent:", by_agent)

---
# 3C - Memory and Observability

## Short-term memory
A follow-up question sent into the same conversation thread as the 3A run. The tool calls made while answering are
counted from the new messages and from the trace: **0 means it was answered from memory**.

In [ ]:
q = ("What 30-day annualised volatility and 90-day downside price band did you find, "
     "and what was the headline sentiment score?")
followup = ask_followup(run_a["graph"], run_a["config"], q)
print("Q:", q, "\n")
print("A:", followup["answer"], "\n")
print("messages already in memory:", followup["messages_in_memory_before"],
      "| tool calls in new messages:", followup["tool_calls_in_messages"] or 0,
      "| tool calls in trace:", followup["tool_calls_in_trace"])

## Persistent memory
The 3B session's final brief is saved to `cache/<TICKER>_<date>.json`. A second request for the same ticker on the
same day detects the file and loads it instead of running any tool.

In [ ]:
path = save_report(report_b, run_b["session_id"])
print("saved:", path.relative_to(tracing.ROOT))
print(json.dumps(json.loads(path.read_text()), indent=2)[:700], "...")

In [ ]:
cached, info = research_with_cache("MSFT", run_pipeline)   # would run the whole pipeline on a cache miss
print("source:", info["source"], "| tool calls:", info["tool_calls"], "| seconds:", info["seconds"],
      "| same report:", cached == report_b)

## Observability: `logs/agent_trace.jsonl`
One JSON line per event. Every tool call records the tool, its inputs, its output truncated to 200 characters, the
wall-clock duration, its status and the calling agent; hand-offs, grounding checks, cache and model-fallback events
are logged too.

In [ ]:
events = trace()
print(f"{len(events)} events in {tracing.TRACE_PATH.name}:",
      dict(pd.Series([e['event'] for e in events]).value_counts()))
calls = pd.DataFrame([e for e in events if e["event"] == "tool_call"])
assert (calls["output"].str.len() <= tracing.MAX_OUTPUT_CHARS).all()
display(calls[["ts", "agent", "tool", "inputs", "output", "duration_ms", "status"]].tail(12))
display(calls.groupby("tool")["duration_ms"].describe()[["count", "mean", "max"]].round(0))

---
# Tests
Offline (no API calls): a scripted model drives the real agent graph to test the observe/decide loop, tool refusal,
error recovery, the tool budget and graceful LLM outages; plus trace fields, report and hand-off schemas, the number
grounding check, the incorporation check, the cache and the follow-up memory.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no", "."],
                   capture_output=True, text=True)
print(r.stdout[-1500:] or r.stderr[-1500:])

---
## Notes and limitations
- **Free-tier quota shapes the design:** 8k tokens/minute and 200k/day for `gpt-oss-120b`. Tool outputs are compact,
  calls are paced, and a daily-quota error falls back to `gpt-oss-20b` (logged) instead of failing.
- **Number grounding is checked, reasoning is not:** every figure in a report or hand-off must appear in a tool
  result (invented figures are sent back for removal), but a correct number can still be interpreted wrongly; an
  LLM-as-judge pass (as in Task 2) would be the next step.
- **News is not cross-checked against data:** a headline can contradict the price data (e.g. "enters bear market"
  while the price is near its 52-week high) and the writer may repeat it.
- **Live results vary between runs:** news, search results and the model's choices change; the outputs above are
  from the run saved in this notebook.
- Not investment advice.